# Конспект: Churn Prediction — Прогнозирование оттока клиентов

В этом конспекте разберём ключевые концепции задачи бинарной классификации на примере прогнозирования оттока клиентов телекоммуникационной компании.

**Что изучим:**
1. Метрика ROC-AUC — почему она лучше accuracy для несбалансированных данных
2. Предобработка данных — нормализация и кодирование признаков
3. LogisticRegression — линейная модель с подбором гиперпараметров
4. CatBoost — градиентный бустинг для категориальных данных
5. Кросс-валидация — надёжная оценка качества модели

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

In [ ]:
# Загружаем данные
data = pd.read_csv('data/train.csv')

# Определяем колонки (из задания)
num_cols = ['ClientPeriod', 'MonthlySpending', 'TotalSpent']
cat_cols = [
    'Sex', 'IsSeniorCitizen', 'HasPartner', 'HasChild', 'HasPhoneService',
    'HasMultiplePhoneNumbers', 'HasInternetService', 'HasOnlineSecurityService',
    'HasOnlineBackup', 'HasDeviceProtection', 'HasTechSupportAccess',
    'HasOnlineTV', 'HasMovieSubscription', 'HasContractPhone',
    'IsBillingPaperless', 'PaymentMethod'
]
feature_cols = num_cols + cat_cols
target_col = 'Churn'

# Исправляем TotalSpent — содержит пробелы вместо чисел
data['TotalSpent'] = pd.to_numeric(data['TotalSpent'], errors='coerce')
data['TotalSpent'] = data['TotalSpent'].fillna(data['TotalSpent'].median())

print(f'Размер данных: {data.shape}')
print(f'Числовых признаков: {len(num_cols)}, категориальных: {len(cat_cols)}')
print(f'Целевая переменная: {target_col}')
data.head(3)

---
## 1. ROC-AUC — метрика для задач классификации

### Почему не accuracy?

Допустим, в нашем датасете 74% клиентов не уходят. Модель, которая **всегда** говорит «не отток», получит accuracy = 74%. Но она бесполезна — она не находит ни одного уходящего клиента!

**ROC-AUC** решает эту проблему:
- Она измеряет, насколько хорошо модель **ранжирует** объекты — присваивает ли она уходящим клиентам более высокие вероятности, чем остающимся
- Не зависит от порога классификации (мы не выбираем, выше какой вероятности считать «отток»)
- 0.5 = случайное угадывание, 1.0 = идеальная модель

### ROC-кривая

ROC-кривая строится так: для каждого возможного порога (от 0 до 1) считается:
- **TPR** (True Positive Rate) = доля правильно найденных объектов класса 1
- **FPR** (False Positive Rate) = доля ложно отнесённых к классу 1 объектов класса 0

ROC-AUC — площадь под этой кривой.

In [ ]:
# Демонстрация: ROC-AUC на простом примере

# Истинные метки
y_true = np.array([0, 0, 0, 1, 1, 1])

# Хорошая модель — уходящим даёт высокие вероятности
y_good = np.array([0.1, 0.2, 0.3, 0.7, 0.8, 0.9])

# Плохая модель — предсказания случайны
y_bad = np.array([0.6, 0.3, 0.8, 0.2, 0.5, 0.4])

# Бесполезная модель — всем одинаково
y_useless = np.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5])

print(f'Хорошая модель:   ROC-AUC = {roc_auc_score(y_true, y_good):.2f}')
print(f'Плохая модель:    ROC-AUC = {roc_auc_score(y_true, y_bad):.2f}')
print(f'Бесполезная:      ROC-AUC = {roc_auc_score(y_true, y_useless):.2f}')

# Визуализация ROC-кривых
fig, ax = plt.subplots(figsize=(7, 6))

for y_pred, label, color in [(y_good, 'Хорошая', 'green'), 
                               (y_bad, 'Плохая', 'orange')]:
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.2f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Случайная (AUC=0.50)', alpha=0.5)
ax.set_xlabel('FPR (False Positive Rate)')
ax.set_ylabel('TPR (True Positive Rate)')
ax.set_title('ROC-кривые: сравнение моделей')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---
## 2. Предобработка данных

Разные модели требуют разной предобработки:

| Что делать | LogisticRegression | CatBoost |
|---|---|---|
| Нормализация числовых | **Обязательно** (StandardScaler) | Не нужна |
| Кодирование категориальных | **Обязательно** (one-hot encoding) | Не нужно (`cat_features`) |
| Заполнение пропусков | Нужно | Нужно |

### 2.1 StandardScaler — нормализация числовых признаков

StandardScaler приводит каждый признак к стандартному распределению:
$$z = \frac{x - \mu}{\sigma}$$

где $\mu$ — среднее, $\sigma$ — стандартное отклонение.

**Зачем?** Логистическая регрессия оптимизирует функцию потерь градиентным спуском. Если признаки имеют разный масштаб (например, `ClientPeriod` ~ 1-72, а `TotalSpent` ~ 0-8600), градиентный спуск будет работать неэффективно.

In [ ]:
# Демонстрация StandardScaler

scaler = StandardScaler()

# До нормализации — разные масштабы
print('ДО нормализации:')
print(data[num_cols].describe().round(1))
print()

# После нормализации — среднее ~0, стандартное отклонение ~1
data_scaled = pd.DataFrame(
    scaler.fit_transform(data[num_cols]),
    columns=num_cols
)
print('ПОСЛЕ нормализации:')
print(data_scaled.describe().round(2))

### 2.2 One-Hot Encoding — кодирование категориальных признаков

Линейные модели не умеют работать со строками (`"Male"`, `"Female"`). One-hot encoding создаёт бинарные колонки для каждого уникального значения:

| Sex | → | Sex_Male |
|-----|---|----------|
| Male | → | 1 |
| Female | → | 0 |

Параметр `drop_first=True` убирает одну колонку — она избыточна (если `Sex_Male=0`, значит `Female`). Это важно для линейных моделей, чтобы избежать мультиколлинеарности.

In [ ]:
# Демонстрация one-hot encoding

# Пример на маленьком датафрейме
example = pd.DataFrame({
    'HasContractPhone': ['Month-to-month', 'One year', 'Two year', 'Month-to-month'],
    'Sex': ['Male', 'Female', 'Male', 'Female']
})
print('Исходные данные:')
print(example)
print()

# One-hot encoding
encoded = pd.get_dummies(example, drop_first=True).astype(int)
print('После pd.get_dummies(drop_first=True):')
print(encoded)
print()

# На реальных данных
X_cat = data[cat_cols].copy()
for col in cat_cols:
    X_cat[col] = X_cat[col].astype(str)

X_cat_encoded = pd.get_dummies(X_cat, drop_first=True).astype(int)
print(f'Категориальных колонок до кодирования: {len(cat_cols)}')
print(f'Бинарных колонок после кодирования: {X_cat_encoded.shape[1]}')

---
## 3. LogisticRegression — линейная модель классификации

### Принцип работы

Логистическая регрессия предсказывает вероятность класса 1 через **сигмоиду** от линейной комбинации признаков:

$$P(y=1|x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}$$

Модель учит веса $w$ и смещение $b$, минимизируя функцию потерь (log-loss).

### Параметр C — регуляризация

- **C** — обратная сила регуляризации: `C = 1/λ`
- **Большой C** (100) — слабая регуляризация → модель может переобучиться
- **Маленький C** (0.001) — сильная регуляризация → модель может недообучиться
- `LogisticRegressionCV` автоматически подбирает лучший C через кросс-валидацию

In [ ]:
# Подготовка данных для LogisticRegression
X = data[feature_cols].copy()
y = data[target_col].copy()

# Приводим категориальные к строкам
for col in cat_cols:
    X[col] = X[col].astype(str)

# Нормализация числовых
scaler = StandardScaler()
X_num_scaled = pd.DataFrame(
    scaler.fit_transform(X[num_cols]),
    columns=num_cols,
    index=X.index
)

# One-hot encoding категориальных
X_cat_encoded = pd.get_dummies(X[cat_cols], drop_first=True).astype(int)

# Объединяем
X_prepared = pd.concat([X_num_scaled, X_cat_encoded], axis=1)
print(f'Итого признаков для LogReg: {X_prepared.shape[1]}')

# Подбор C через LogisticRegressionCV
Cs = [100, 10, 1, 0.1, 0.01, 0.001]

log_reg_cv = LogisticRegressionCV(
    Cs=Cs,
    scoring='roc_auc',   # Подбираем C по ROC-AUC
    refit=True,           # Переобучить на всех данных с лучшим C
    cv=5,                 # 5-fold кросс-валидация
    random_state=42,
    max_iter=1000
)
log_reg_cv.fit(X_prepared, y)

# Результаты для каждого C
cv_scores = log_reg_cv.scores_[1]  # scores для класса 1
mean_scores = cv_scores.mean(axis=0)

print(f'\nЛучший C: {log_reg_cv.C_[0]}')
print(f'\nROC-AUC для каждого C:')
for c, score in zip(Cs, mean_scores):
    marker = ' ← лучший' if c == log_reg_cv.C_[0] else ''
    print(f'  C={c:>6}: {score:.4f}{marker}')

---
## 4. CatBoost — градиентный бустинг

### Принцип работы

Градиентный бустинг строит **ансамбль деревьев решений** последовательно: каждое следующее дерево учится на ошибках предыдущих.

**CatBoost** (Categorical Boosting) — разработка Яндекса с двумя ключевыми особенностями:
1. **Ordered target encoding** для категориальных признаков — не нужен ручной one-hot encoding
2. **Ordered boosting** — борьба с утечкой данных при обучении

### Ключевые гиперпараметры

| Параметр | Что делает | Влияние |
|----------|-----------|---------|
| `n_estimators` | Количество деревьев | Больше деревьев → дольше обучение, риск переобучения |
| `learning_rate` | Скорость обучения | Меньше lr → нужно больше деревьев, но стабильнее |
| `depth` | Глубина дерева | Глубже → сложнее модель, больше риск переобучения |

**Закономерность**: маленький `learning_rate` + много `n_estimators` = медленное, но стабильное обучение. Большой `learning_rate` + много деревьев = переобучение.

In [ ]:
# CatBoost — минимальный пример

# Подготовка данных (БЕЗ нормализации и one-hot — CatBoost справится сам)
X_cb = data[feature_cols].copy()
y_cb = data[target_col].copy()

for col in cat_cols:
    X_cb[col] = X_cb[col].astype(str)

# Разделение на train/valid
X_train, X_valid, y_train, y_valid = train_test_split(
    X_cb, y_cb, test_size=0.2, random_state=42, stratify=y_cb
)

# Обучение CatBoost с параметрами по умолчанию
cb_baseline = CatBoostClassifier(random_state=42, verbose=0)
cb_baseline.fit(X_train, y_train, cat_features=cat_cols)

y_pred_proba = cb_baseline.predict_proba(X_valid)[:, 1]
baseline_auc = roc_auc_score(y_valid, y_pred_proba)
print(f'CatBoost (defaults) ROC-AUC: {baseline_auc:.4f}')
print('→ Без нормализации, без one-hot — и уже хороший результат!')

In [ ]:
# Подбор гиперпараметров: n_estimators и learning_rate
# Посмотрим, как они взаимодействуют

n_estimators_list = [100, 500, 1000]
learning_rate_list = [0.01, 0.05, 0.1, 0.2]

results = []
for n_est in n_estimators_list:
    for lr in learning_rate_list:
        model = CatBoostClassifier(
            n_estimators=n_est,
            learning_rate=lr,
            random_state=42,
            verbose=0
        )
        model.fit(X_train, y_train, cat_features=cat_cols)
        auc = roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])
        results.append({'n_estimators': n_est, 'learning_rate': lr, 'roc_auc': auc})

results_df = pd.DataFrame(results)
pivot = results_df.pivot(index='n_estimators', columns='learning_rate', values='roc_auc')
print('ROC-AUC для разных комбинаций (n_estimators × learning_rate):')
print(pivot.round(4))

best_row = results_df.loc[results_df['roc_auc'].idxmax()]
print(f'\nЛучшая комбинация: n_estimators={int(best_row["n_estimators"])}, '
      f'lr={best_row["learning_rate"]}, ROC-AUC={best_row["roc_auc"]:.4f}')

---
## 5. Кросс-валидация — надёжная оценка модели

### Проблема однократного разбиения

Когда мы делим данные на train/valid один раз (`train_test_split`), результат зависит от того, **какие** объекты попали в valid. Может повезти — или не повезти.

### Кросс-валидация (K-Fold)

Данные делятся на K частей (фолдов). Модель обучается K раз:
- Каждый раз одна часть — тестовая, остальные K-1 — обучающие
- Итог: K оценок → берём **среднее** и **стандартное отклонение**

Это даёт стабильную оценку качества.

```
Fold 1: [TEST] [train] [train] [train] [train]
Fold 2: [train] [TEST] [train] [train] [train]
Fold 3: [train] [train] [TEST] [train] [train]
Fold 4: [train] [train] [train] [TEST] [train]
Fold 5: [train] [train] [train] [train] [TEST]
```

In [ ]:
# Демонстрация кросс-валидации через cross_val_score

# LogisticRegression с кросс-валидацией
log_reg = LogisticRegression(C=10, max_iter=1000, random_state=42)

cv_scores = cross_val_score(
    log_reg, X_prepared, y,
    cv=5,              # 5 фолдов
    scoring='roc_auc'  # Метрика
)

print('ROC-AUC по фолдам:', [f'{s:.4f}' for s in cv_scores])
print(f'Среднее: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print()
print('Стандартное отклонение маленькое → оценка стабильная')

---
## 6. Predict_proba vs Predict

Важное различие для задач, оцениваемых по ROC-AUC:

- `model.predict(X)` → возвращает **метки** (0 или 1)
- `model.predict_proba(X)` → возвращает **вероятности** каждого класса

Для ROC-AUC нужны именно **вероятности класса 1** — `predict_proba(X)[:, 1]`.

In [ ]:
# Разница между predict и predict_proba

log_reg.fit(X_prepared, y)

# predict — жёсткие метки
labels = log_reg.predict(X_prepared[:5])
print('predict (метки):')
print(labels)
print()

# predict_proba — вероятности [P(class=0), P(class=1)]
probas = log_reg.predict_proba(X_prepared[:5])
print('predict_proba (вероятности):')
print(probas.round(3))
print()

# Для submission нужна только вероятность класса 1
print('Вероятность класса 1 (для submission):')
print(probas[:, 1].round(3))

---
## 7. Сравнение подходов

| Аспект | LogisticRegression | CatBoost |
|--------|-------------------|----------|
| Предобработка | StandardScaler + one-hot encoding | Только `cat_features` |
| Скорость обучения | Быстро | Медленнее |
| Интерпретируемость | Высокая (веса признаков) | Средняя (feature importance) |
| Нелинейные зависимости | Не моделирует | Моделирует |
| Типичный ROC-AUC на этих данных | ~0.845 | ~0.855+ |

**Вывод:** CatBoost даёт лучший результат при меньших усилиях на предобработку. Но LogisticRegression — отличный baseline, быстрый и интерпретируемый.

---
## 8. Попробуй сам

### Задание 1: Визуализация дисбаланса классов
Построй bar-диаграмму распределения целевой переменной `Churn`. Сколько процентов составляет класс 1?

> **Подсказка:** используй `data['Churn'].value_counts()` и `plt.bar()`

In [ ]:
# Задание 1: Твой код здесь
# churn_counts = data['Churn'].value_counts()
# plt.bar(...)
# print(f'Доля класса 1: {... * 100:.1f}%')


### Задание 2: Подбор параметра C вручную
Обучи `LogisticRegression` с разными значениями C (0.001, 0.01, 0.1, 1, 10, 100) и выведи ROC-AUC на валидации для каждого. Какой C лучший?

> **Подсказка:** используй `train_test_split`, цикл по значениям C, `roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])`

In [ ]:
# Задание 2: Твой код здесь
# X_tr, X_val, y_tr, y_val = train_test_split(X_prepared, y, test_size=0.2, random_state=42)
# for C in [0.001, 0.01, 0.1, 1, 10, 100]:
#     model = LogisticRegression(C=C, max_iter=1000, random_state=42)
#     model.fit(X_tr, y_tr)
#     auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
#     print(f'C={C:>6}: ROC-AUC = {auc:.4f}')


### Задание 3: Feature Importance в CatBoost
Обучи CatBoost и выведи важность признаков. Какие 5 признаков самые важные?

> **Подсказка:** после `model.fit()` используй `model.get_feature_importance()` и `feature_names=model.feature_names_`

In [ ]:
# Задание 3: Твой код здесь
# importances = cb_baseline.get_feature_importance()
# feat_imp = pd.Series(importances, index=cb_baseline.feature_names_).sort_values(ascending=False)
# print('Топ-5 важных признаков:')
# print(feat_imp.head(5))
# feat_imp.head(10).plot(kind='barh')
# plt.title('Feature Importance (CatBoost)')
# plt.show()


---
## Полезные ссылки

- [LogisticRegressionCV (sklearn)](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegressionCV.html) — подбор C через кросс-валидацию
- [StandardScaler (sklearn)](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) — нормализация признаков
- [roc_auc_score (sklearn)](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html) — вычисление ROC-AUC
- [CatBoostClassifier](https://catboost.ai/en/docs/concepts/python-reference_catboostclassifier) — документация CatBoost
- [pd.get_dummies (pandas)](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) — one-hot encoding